In [5]:
import pandas as pd
# --- NOUVELLE SYNTAXE API EVIDENTLY 0.7+ ---
from evidently import Report
from evidently.presets import DataDriftPreset
# -------------------------------------------
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("1. Initialisation des chemins...")
PROJECT_ROOT = Path.cwd().parent
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DRIFT_DIR = PROJECT_ROOT / "drift"

# Création du dossier drift s'il n'existe pas
DRIFT_DIR.mkdir(parents=True, exist_ok=True)

print("2. Chargement des échantillons de données (Train vs Test)...")
# On prend 5000 lignes au hasard pour ne pas saturer la RAM
train_data = pd.read_csv(DATA_PROCESSED / "train_final.csv").sample(n=5000, random_state=42)
test_data = pd.read_csv(DATA_PROCESSED / "test_final.csv").sample(n=5000, random_state=42)

# On retire la cible (TARGET) et l'ID car on veut mesurer la dérive des caractéristiques clientes, pas des résultats
cols_to_drop = ['TARGET', 'SK_ID_CURR']
reference_data = train_data.drop(columns=[c for c in cols_to_drop if c in train_data.columns])
current_data = test_data.drop(columns=[c for c in cols_to_drop if c in test_data.columns])

# Alignement strict des colonnes (pour éviter les erreurs d'encodage)
common_cols = list(set(reference_data.columns) & set(current_data.columns))
reference_data = reference_data[common_cols]
current_data = current_data[common_cols]

print(f"Dimensions de référence (Train) : {reference_data.shape}")
print(f"Dimensions actuelles (Test) : {current_data.shape}")

print("\n3. Calculs statistiques du Drift en cours avec Evidently AI (Patientez ~1 min)...")
# 1. On définit le modèle du rapport (la coquille vide)
report_template = Report(metrics=[DataDriftPreset()])

# 2. LA CORRECTION EST ICI : On stocke les résultats dans une nouvelle variable (my_eval)
my_eval = report_template.run(reference_data=reference_data, current_data=current_data)

print("\n4. Sauvegarde du tableau de bord HTML...")
report_path = DRIFT_DIR / "data_drift_report.html"

# 3. On sauvegarde le fichier HTML à partir des résultats, avec save_html()
my_eval.save_html(str(report_path))

print(f"✅ Terminé ! Allez ouvrir le fichier : {report_path}")

1. Initialisation des chemins...
2. Chargement des échantillons de données (Train vs Test)...
Dimensions de référence (Train) : (5000, 176)
Dimensions actuelles (Test) : (5000, 176)

3. Calculs statistiques du Drift en cours avec Evidently AI (Patientez ~1 min)...

4. Sauvegarde du tableau de bord HTML...
✅ Terminé ! Allez ouvrir le fichier : /home/alouiyaz/projects/Qualité_controle_données/drift/data_drift_report.html
